# Overview

`delnx` is a Python package for differential expression analysis of single-cell RNA-seq data. It provides a unified interface to several ways of performing differential expression analysis with (generalized) linear models. Most methods are implemented in [JAX](https://jax.readthedocs.io/en/latest/) for GPU-accelerated DE testing, with [statsmodels](https://www.statsmodels.org/stable/index.html) as a fallback for binomial GLMs. To get you started, here's a basic example of how to use `delnx` for differential expression analysis:

In [1]:
import delnx as dx
import scanpy as sc


# Load example data
adata = sc.read_h5ad("data/GLI3_KO_45d.h5ad")
adata.layers["counts"] = adata.X.copy()  # Store raw counts in a separate layer
adata.obs["GLI3_KO"] = adata.obs["GLI3_KO"].astype(str)  # Ensure string type for design matrix

# Some basic preprocessing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(adata)

AnnData object with n_obs × n_vars = 22410 × 18653
    obs: 'organoid', 'GLI3_KO', 'cell_type'
    uns: 'log1p'
    obsm: 'pca', 'umap'
    layers: 'counts'


In [2]:
# Run differential expression analysis between conditions (here knockout vs. control)
de_results = dx.tl.de(
    adata,
    method="lr",  # DE method: "lr" for logistic regression
    condition_key="GLI3_KO",  # Condition key for DE analysis
    reference="True",  # Reference level
    contrast="GLI3_KO[T.False]",  # Test knockout vs control
)

print(de_results)

INFO     Inferred data type: lognorm                                                                               


INFO     Running DE for 16199 features                                                                             


  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:03<00:21,  3.10s/it]

 25%|██▌       | 2/8 [00:05<00:14,  2.47s/it]

...

 75%|███████▌  | 6/8 [00:13<00:04,  2.12s/it]

 88%|████████▊ | 7/8 [00:15<00:02,  2.09s/it]

100%|██████████| 8/8 [00:17<00:00,  2.13s/it]

100%|██████████| 8/8 [00:17<00:00,  2.20s/it]

      feature    log2fc      coef          stat          pval          padj
0      SHISA3 -0.478362 -0.331575  3.397817e+02  1.000000e-50  7.534419e-49
1        FZD5 -0.441296 -0.305883  3.975457e+02  1.000000e-50  7.534419e-49
2      CRYBA1 -0.410040 -0.284218  2.466093e+02  1.000000e-50  7.534419e-49
3      NKX6-1 -0.409135 -0.283591  3.141194e+02  1.000000e-50  7.534419e-49
4       SFTA3 -0.353181 -0.244807  1.449701e+03  1.000000e-50  7.534419e-49
...       ...       ...       ...           ...           ...           ...
16194    ECM1 -0.000050 -0.000035  5.194038e-07  9.994250e-01  9.996718e-01
16195  KLHL11 -0.000031 -0.000021  3.173746e-07  9.995505e-01  9.997201e-01
16196   MED14 -0.000008 -0.000006  2.555619e-07  9.995967e-01  9.997201e-01
16197    PYGB  0.000002  0.000001  2.436492e-08  9.998755e-01  9.999372e-01
16198  TRIM58  0.000004  0.000002  2.388738e-09  9.999610e-01  9.999610e-01

[16199 rows x 6 columns]


What you get back from {func}`~delnx.tl.de` is a {class}`~pandas.DataFrame` with the results of the differential expression analysis. The rows are genes and the columns are the results of the DE testing: `log2fc`, `coef`, `stat`, `pval`, and `padj`.

## Grouping
Often, we want to split the dataset into groups, to perform testing e.g. within each cell type. `delnx` provides the {func}`~delnx.tl.grouped` wrapper for this. It runs any DE function separately for each group and re-corrects p-values across all groups.

In [3]:
# Run differential expression analysis within groups
de_results = dx.tl.grouped(
    dx.tl.de,
    adata,
    group_key="cell_type",  # Group by cell type
    method="lr",  # DE method: "lr" for logistic regression
    condition_key="GLI3_KO",  # Condition key for DE analysis
    reference="True",  # Reference level
    contrast="GLI3_KO[T.False]",  # Test knockout vs control
)

print(de_results)

INFO     Running DE for group: mesen_npc                                                                           


INFO     Inferred data type: lognorm                                                                               


INFO     Running DE for 15443 features                                                                             


  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:00<00:04,  1.47it/s]

...

  0%|          | 0/7 [00:00<?, ?it/s]

 14%|█▍        | 1/7 [00:00<00:02,  2.86it/s]

100%|██████████| 7/7 [00:00<00:00, 11.80it/s]

100%|██████████| 7/7 [00:00<00:00, 10.39it/s]

      feature    log2fc      coef          stat          pval          padj  \
0       RGS20 -0.438849 -0.304187  1.061743e+02  4.705923e-23  8.590426e-21   
1       QPCTL -1.223741 -0.848233  9.882085e+01  1.126694e-21  1.822624e-19   
2       ASF1B -0.927407 -0.642829  9.369446e+01  1.052890e-20  1.594351e-18   
3        PON2 -0.299645 -0.207698  9.349043e+01  1.151250e-20  1.740644e-18   
4        POMC -0.722494 -0.500794  8.947009e+01  6.728782e-20  9.481960e-18   
...       ...       ...       ...           ...           ...           ...   
99482  OSBPL8  0.000012  0.000008  8.110453e-07  9.992815e-01  9.996337e-01   
99483   EGFL8  0.000100  0.000069  5.362082e-07  9.994158e-01  9.997055e-01   
99484   PTPN7 -0.000056 -0.000039  2.285110e-07  9.996186e-01  9.997794e-01   
99485   DEAF1 -0.000008 -0.000006  1.727580e-07  9.996684e-01  9.998091e-01   
99486  SLFN13  0.000030  0.000021  1.247965e-08  9.999109e-01  9.999209e-01   

           group  
0         ctx_ex  
1         ctx

Now the result has one additional column, `group`, which indicates the group the gene was tested in.

## Pseudo-bulking
It is often advisable to not test on the single-cell level, but to aggregate the data to a (pseudo-)bulk level first. This better accounts for variation between actual biological replicates. `delnx` provides a thin wrapper around the [decoupler](https://decoupler.readthedocs.io/en/latest) function to do this:

In [4]:
adata_pb = dx.pp.pseudobulk(
    adata,
    sample_key="organoid",  # Sample key for pseudobulk aggregation (the biological replicate)
    group_key="cell_type",  # Group key for pseudobulk aggregation
    n_pseudoreps=2,  # Optionally, the data can be split into multiple pseudoreplicates. This can be useful if the number of actual biological replicates is low.
    layer="counts",  # Layer to use for pseudobulk aggregation, e.g. "counts" or None for adata.X
    mode="sum",
)

print(adata_pb)

AnnData object with n_obs × n_vars = 81 × 16199
    obs: 'psbulk_replicate', 'cell_type', 'organoid', 'GLI3_KO', 'psbulk_cells', 'psbulk_counts'
    layers: 'psbulk_props'


## Available Methods
`delnx` provides various methods for performing differential expression analysis, which you can specify with the `method` argument. The available methods for {func}`~delnx.tl.de` are:

- `"lr"`: Logistic regression with likelihood ratio test. Recommended for log-normalized single-cell data.

- `"anova"`: ANOVA based on linear model. Recommended for log-normalized or scaled data.

- `"anova_residual"`: Linear model with residual F-test. Recommended for log-normalized or scaled data.

- `"binomial"`: Binomial GLM likelihood ratio test. Recommended for binary data such as ATAC-seq.

For **count data** (raw RNA-seq counts), use {func}`~delnx.tl.nb_fit` + {func}`~delnx.tl.nb_test` which implement GPU-accelerated negative binomial GLMs with quasi-likelihood dispersion shrinkage. See {doc}`nb` for more details.

For **fast cluster markers**, use {func}`~delnx.tl.rank_de` which uses AUROC-based one-vs-all testing.

All methods in {func}`~delnx.tl.de` support R-style formulas via the `formula` parameter. This allows you to include covariates directly:

```python
results = dx.tl.de(adata, formula="~ GLI3_KO + batch", contrast="GLI3_KO[T.False]")
```